# [한 번 해보기] 음식 리뷰 데이터 활용 유사도 검색

- corpus -> embedding vector -> 유사도 기반 검색
    - Summary(title), Text(content) --> Content
    - Content -> Embedding Vector1 (.csv 저장)
    - 사용자가 입력 (best coffee) -> Embedding Vector2
    - 두 개의 유사도를 검색해서 유사도가 높은 것을 찾으면 됨
- OpenAPI 임베딩 모델 호출해서 Inference를 쓰려면 API key값을 집어넣어야 함!!!
    - *깃허브 키는 노출 되어서는 안 돼요~~~!!! 유의하기!*

### 1. 리뷰 텍스트 불러오기

In [17]:
import pandas as pd
import numpy as np

### 2. Embedding Vector 1 (Summary, title)

In [18]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
from openai import OpenAI
import pandas as pd

client = OpenAI()

df = pd.read_csv('./data/fine_food_reviews_1k.csv')
df.head()

,Unnamed: 0,Time,ProductId,UserId,Score,Summary,Text
0,0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...
1,1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos..."
2,2,1351123200,B000JMBE7M,AQX1N6A51QOKG,4,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...
3,3,1351123200,B004AHGBX4,A2UY46X0OSNVUQ,3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...
4,4,1351123200,B001BORBHO,A1AFOYZ9HSM2CZ,5,Happy with the product,My dog was suffering with itchy skin. He had ...


In [20]:
df['Summary'] = df['Summary'].astype(str)
df['Text'] = df['Text'].astype(str)

# 제목과 본문을 하나의 문서로 결합
df['Content'] = (df['Summary'].str.strip() + ' ' + df['Text'].str.strip()).str.strip()
df[['Summary', 'Text', 'Content']].head()

,Summary,Text,Content
0,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,where does one start...and stop... with a tre...
1,Arrived in pieces,"Not pleased at all. When I opened the box, mos...",Arrived in pieces Not pleased at all. When I o...
2,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...,"It isn't blanc mange, but isn't bad . . . I'm ..."
3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...,These also have SALT and it's not sea salt. I ...
4,Happy with the product,My dog was suffering with itchy skin. He had ...,Happy with the product My dog was suffering wi...


### 3. 임베딩 추출 함수 정의

In [25]:
# 3. 임베딩 추출 함수 정의
def text_to_embedding(text_list, model='text-embedding-3-small'):
    # 입력받은 text_list를 순회하며 개행문자 제거
    cleaned_texts = [text.replace('\n', ' ') for text in text_list]
    
    response = client.embeddings.create(
        model=model,
        input=cleaned_texts
    )
    
    # response.data에서 차례대로 임베딩 벡터값만 추출하여 리스트로 반환
    return [data.embedding for data in response.data]

### 3. 리뷰 문서 임베딩

In [27]:
doc_embeddings = text_to_embedding(df['Content'].tolist())

# DataFrame에 저장
emb_dim = len(doc_embeddings[0])

# 필요하면 CSV로 저장 가능
embedding_df = pd.DataFrame({
    'Summary': df['Summary'],
    'Text': df['Text'],
    'Content': df['Content'],
    'embedding': doc_embeddings
})

embedding_df.head()

,Summary,Text,Content,embedding
0,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,where does one start...and stop... with a tre...,"[0.031951904296875, -0.017791748046875, -0.028..."
1,Arrived in pieces,"Not pleased at all. When I opened the box, mos...",Arrived in pieces Not pleased at all. When I o...,"[0.010101318359375, 0.0401611328125, -0.040100..."
2,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...,"It isn't blanc mange, but isn't bad . . . I'm ...","[0.0015697479248046875, 0.00426483154296875, -..."
3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...,These also have SALT and it's not sea salt. I ...,"[-0.01520538330078125, 0.01270294189453125, -0..."
4,Happy with the product,My dog was suffering with itchy skin. He had ...,Happy with the product My dog was suffering wi...,"[0.0017147064208984375, -0.06781005859375, 0.0..."


### 4. 사용자 질의 임베딩 만들기

In [35]:
query = input("검색어를 입력하세요: ").strip()

if query:
    query_embedding = text_to_embedding([query])[0]

    print(f'질의: {query}')
    print(f'질의 임베딩 차원 수: {len(query_embedding)}')
else:
    print("검색어가 비어 있습니다.")

질의: coffee
질의 임베딩 차원 수: 1536


### 3. 코사인 유사도

In [36]:
#!pip install sklearn

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

doc_matrix = np.array(doc_embeddings)
query_vector = np.array(query_embedding).reshape(1, -1)

similarities = cosine_similarity(query_vector, doc_matrix)[0]
df['similarity'] = similarities

# 유사도가 높은 상위 5개 리뷰 확인
top_k = 5
result = df[['Summary', 'Text', 'similarity']].sort_values(by='similarity', ascending=False).head(top_k)
result

,Summary,Text,similarity
30,Better than you-know-who's coffee...,"So my wife is a latte freak, and nursing, so d...",0.406516
170,"Coffee great, seller not so great.","We drink Green Mountain coffee all the time, a...",0.406354
77,Great alternative to daily Starbucks trips,My wife and I are big fans of hot morning drin...,0.402663
251,Great Coffee,I have a coffee maker that grinds my coffee be...,0.399869
553,Love My Senseo!,I I haven't had a bad cup of coffee yet. So f...,0.398130


### 4. 임베딩 결과 저장하기

In [34]:
# 리스트 형태 임베딩을 문자열로 저장하면 CSV로 보관 가능
save_df = embedding_df.copy()
save_df['embedding'] = save_df['embedding'].apply(lambda x: str(x))
save_path = './data/fine_food_reviews_with_embeddings.csv'
save_df.to_csv(save_path, index=False)